# 04. XGBoost Model Training — Baseline (no preprocessing)

This notebook trains an **XGBoost** classifier with **default settings** on all **103 signal features**, without any preprocessing, to get a first baseline score.

It uses exactly the same data and steps as `04d_lightgbm_baseline.ipynb`, `04a_logistic_regression_baseline.ipynb` and `04b_random_forest_baseline.ipynb`, so the results can be compared.

Why no preprocessing is acceptable here:
- **Scaling** is not needed: XGBoost is made of decision trees, which split on value thresholds, so feature scale does not matter.
- **Imputation** is not needed: the 103 features have no missing values.
- **Feature selection** is not done: this is the *full feature set* baseline.

Rules followed:
- Train on `train.csv` only, score on `val.csv` only.
- `test.csv` is **not loaded** in this notebook.
- `record_id`, `patient_id` and `figo_label` are not used as input features.
- Positive class = abnormal (`figo_label = 1`). Threshold = 0.5 (default).

## Step 1: Setup and imports

In [1]:
import sys
sys.path.append('../..')

import time
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score, confusion_matrix

from config import config
from src.data_preprocessing import get_feature_columns

## Step 2: Load the training and validation sets

The test set stays untouched until the final evaluation.

In [2]:
# Load train and validation sets only (test set stays untouched)
train_df = pd.read_csv(config.PROCESSED_TRAIN_FILE)
val_df = pd.read_csv(config.PROCESSED_VAL_FILE)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Train shape: (14540, 106)
Validation shape: (3115, 106)


## Step 3: Separate features (X) and target (y)

In [3]:
# Get the 103 signal features (IDs and target are excluded)
feature_cols = get_feature_columns(train_df)

X_train = train_df[feature_cols]
y_train = train_df[config.TARGET_COLUMN]

X_val = val_df[feature_cols]
y_val = val_df[config.TARGET_COLUMN]

print("Train:", X_train.shape, " Validation:", X_val.shape)

Number of feature columns found: 103
Train: (14540, 103)  Validation: (3115, 103)


## Step 4: Create and train the XGBoost model

Default hyperparameters; only the random seed is fixed for reproducibility.

In [4]:
# Create XGBoost with default settings (only the seed is fixed)
model = XGBClassifier(random_state=config.RANDOM_STATE)

# Train on the training set only
start_time = time.time()
model.fit(X_train, y_train)
train_seconds = time.time() - start_time

print("Training time (seconds):", round(train_seconds, 2))

# Show the main default settings that were used
params = model.get_params()
print("n_estimators: ", params["n_estimators"])
print("max_depth:    ", params["max_depth"])
print("learning_rate:", params["learning_rate"])
print("(None means XGBoost uses its own internal default)")

Training time (seconds): 3.48
n_estimators:  None
max_depth:     None
learning_rate: None
(None means XGBoost uses its own internal default)


## Step 5: Predict on the validation set

In [5]:
# Class predictions (threshold 0.5 by default)
val_pred = model.predict(X_val)

# Probability of abnormal (1), needed for ROC-AUC
val_prob = model.predict_proba(X_val)[:, 1]

## Step 6: Validation metrics

Positive class = abnormal (`figo_label = 1`).

In [6]:
print("Accuracy:         ", round(accuracy_score(y_val, val_pred), 4))
print("Precision:        ", round(precision_score(y_val, val_pred), 4))
print("Recall:           ", round(recall_score(y_val, val_pred), 4))
print("F1-score:         ", round(f1_score(y_val, val_pred), 4))
print("ROC-AUC:          ", round(roc_auc_score(y_val, val_prob), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_val, val_pred), 4))

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_val, val_pred))

Accuracy:          0.94
Precision:         0.9429
Recall:            0.9457
F1-score:          0.9443
ROC-AUC:           0.9871
Balanced accuracy: 0.9395
Confusion matrix [[TN, FP], [FN, TP]]:
[[1343   96]
 [  91 1585]]


## Step 7: Check for overfitting

If the training F1 is much higher than the validation F1, the model is overfitting.

In [7]:
# Training F1, compared with validation F1 from Step 6
train_pred = model.predict(X_train)
print("Training F1:  ", round(f1_score(y_train, train_pred), 4))
print("Validation F1:", round(f1_score(y_val, val_pred), 4))

Training F1:   1.0
Validation F1: 0.9443


## Step 8: Top 15 most important features

XGBoost gives each feature an importance score (how much it helped the trees separate normal from abnormal).
This is only a first look, learned from the training set. It is **not** the feature selection of notebook 03.

In [8]:
# Put the feature names and their importance scores into a table
importance_table = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
})

# Sort from most to least important
importance_table = importance_table.sort_values("importance", ascending=False)

print(importance_table.head(15).to_string(index=False))

     feature  importance
  FU_V_WCTTF    0.172206
       F_D_N    0.054590
 FU_RE_WCTTF    0.047574
  F_LPF_ST11    0.039168
    U_V_STTF    0.028766
        F_SD    0.025914
    F_PSD_MF    0.022199
   F_K_CWTTF    0.020721
   F_M_CWTTF    0.020353
  F_RE_CWTTF    0.020022
FU_LPF_WCT11    0.017803
   F_S_CWTTF    0.016821
   F_V_CWTTF    0.015731
        U_MN    0.014368
      F_LD_N    0.014084
